In [0]:
# ---===: How many dt= partitions did you generate for t_Client?
OUTPUT_PATH = "/Volumes/bfsi_lakehouse/raw/synthetic_data/"
# print(f"{OUTPUT_PATH}/t_Client")


files = dbutils.fs.ls(f"{OUTPUT_PATH}/t_Client")
print(len(files))

In [0]:
# ---===: For t_Client/dt=2024-01-01/, how many batch= subfolders? How many parquet files inside batch=1?
OUTPUT_PATH = "/Volumes/bfsi_lakehouse/raw/synthetic_data/"
files = dbutils.fs.ls(f"{OUTPUT_PATH}/t_Client/dt=2024-01-01/batch=1/")

op = 0
for items in files:
    if items.name.endswith(".parquet"):
        op += 1
        print(items.name)
    
print(op)

# print(len(files))

In [0]:
dbutils.fs.head("dbfs:/Volumes/bfsi_lakehouse/raw/synthetic_data/t_Client/dt=2024-01-01/batch=1/part-00000-tid-3911351928343475580-305180ab-9d14-4276-a641-409d941ca1b1-134-1.c000.snappy.parquet", 200)

In [0]:
 # ---===: Find out total file count across all of t_Client (all dates, all batches)
OUTPUT_PATH = "/Volumes/bfsi_lakehouse/raw/synthetic_data/"
files_dir1 = dbutils.fs.ls(f"{OUTPUT_PATH}/t_Client")
files_dt = []

# Day wise files
for i in files_dir1:
    files_dt.append(i.path)
    # print(i.path)

# Day-Batch wise files
files_dt_batch_path = []
for i in files_dt:
    files_dt_batch = dbutils.fs.ls(i)
    for j in files_dt_batch:
        files_dt_batch_path.append(j.path)
        # print(j.path)

# Day-Batch wise all inside files
all_files_path = []
for i in files_dt_batch_path:
    batch_files = dbutils.fs.ls(i)
    for j in batch_files:
        if j.name.endswith(".parquet"):
            all_files_path.append(i.path)
        

print(len(all_files_path))

In [0]:
# Utility Function to Get Table wise file counts from raw data.
from collections import Counter

VOLUMN_PATH = "/Volumes/bfsi_lakehouse/raw/synthetic_data/"
TABLE_NAMES = ["t_Client", "t_AccountCustomer", "t_Loan", "t_LoanInstallment", "t_AccountTrx"]

def count_files_by_table(base_path: str, table_names: list, file_type: str) -> dict:
    # ---: Path Pattern & Dynamic path creation
    table_path = [f"{base_path}{tables}" for tables in table_names]
    # print(table_path)

    paths = []
    date_layer = [dbutils.fs.ls(path) for path in table_path]
    for main_table in date_layer:   # Main Table files
        for dates in main_table:    # Date wise files
            date_batch=dbutils.fs.ls(dates.path)
            for files in date_batch:    # batch wise files
                for actual_files in dbutils.fs.ls(files.path):
                    if actual_files.name.endswith(file_type):   # actual parquet files
                        # print(actual_files.path)
                        paths.append(actual_files.path)

    tables = []
    for i in paths:
        tables.append(i.split("/")[5])
    
    counter = Counter(tables)
    return dict(counter)


result = count_files_by_table(
    base_path = VOLUMN_PATH,
    table_name = TABLE_NAMES,
    file_type = '.parquet'
)
print(result)

In [0]:
# Utility Function to Get Table wise file counts from raw data.

VOLUMN_PATH = "/Volumes/bfsi_lakehouse/raw/synthetic_data/"
TABLE_NAMES = ["t_Client", "t_AccountCustomer", "t_Loan", "t_LoanInstallment", "t_AccountTrx"]


def count_files_by_table(base_path: str, table_names: list, file_type: str) -> dict:
    result = {}
    for table in table_names:
        result[table] = 0
        main_path = base_path + table
        #    print(main_path)
        for date in dbutils.fs.ls(main_path):
            #    print(date.path)
            for batch in dbutils.fs.ls(date.path):
                #    print(batch.path)
                for parquet_path in dbutils.fs.ls(batch.path):
                    if parquet_path.name.endswith(file_type):
                        #    print(parquet_path.path)
                        result[table] += 1
    return result
               
            


result = count_files_by_table(
    base_path = VOLUMN_PATH,
    table_names = TABLE_NAMES,
    file_type = '.parquet'
)
print(result)